# ETL Data Explorer

This notebook explores the unified ETL (Extract, Transform, Load) Pipeline for stock data.

**Version:** 1.0.0 | **Model Version:** v9_9

## Pipeline Stages
1. **Extract** - Load from DB with CSV fallback
2. **Transform** - Normalize, validate, sanitize, impute (6-step)
3. **Load** - Quality validation and finalization

## 16 Feature Categories (196 Features)
- Momentum & Technical, Valuation Ratios, Profitability, Quality & Risk
- Cash Flow, Capital Allocation, Analyst Sentiment, Market Sentiment
- Leverage & Liquidity, Temporal Patterns, Composite Scores, Growth Metrics
- Efficiency Ratios, Employee Productivity, Balance Sheet, Revenue Forecasting

## Feature Engineering API (`build_features`)

**Business Goal:** Engineer comprehensive financial features including valuation ratios, 
profitability metrics, quality indicators, and sector-specific features to maximize model predictive power.

**Key Objectives:**
- Engineer valuation ratios (P/E, P/B, EV/EBITDA, PEG)
- Engineer profitability features (margins, ROE, ROA, ROIC)
- Create momentum and technical indicators
- Engineer analyst quality features
- Create accounting quality scores (Altman Z, Piotroski F)
- Build sector-relative features
- Create interaction features


In [1]:
# ============================================================================
# Cell 1: Configuration & Setup
# ============================================================================
import json
import math
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# NumPy compatibility layer
if not hasattr(np, '_ARRAY_API'):
    class MockArrayAPI:
        pass


    np._ARRAY_API = MockArrayAPI()

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# SQLAlchemy check
try:
    from sqlalchemy import create_engine, text

    HAVE_SQLALCHEMY = True
except ImportError:
    HAVE_SQLALCHEMY = False

# Project paths
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CACHE_DIR = PROJECT_ROOT / '.cache'

(OUTPUT_DIR / 'eda').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'preprocessing').mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finance_ml.notebook_config import NotebookConfig

CFG = NotebookConfig(
        have_finance_prediction=True,
        have_database_connection=False,
        have_advanced_analytics=True,
        have_dim_reduction=False,
        debug_mode=False,
        )

# Section 17 Style Guidelines
plt.style.use('dark_background')
sns.set_palette('husl')
PLOTLY_TEMPLATE = 'plotly_dark'
COLOR_PALETTE = {
    'primary': '#375a7f',
    'success': '#00bc8c',
    'warning': '#f39c12',
    'danger': '#e74c3c',
    'info': '#3498db',
    'neutral': '#adb5bd',
    }

# Database URL
DB_URL_ENV = os.getenv('DB_URL')
if DB_URL_ENV and DB_URL_ENV.startswith('jdbc:'):
    DB_URL = DB_URL_ENV.replace('jdbc:postgresql://', 'postgresql+psycopg2://postgres:@')
elif DB_URL_ENV:
    DB_URL = DB_URL_ENV
else:
    DB_URL = 'postgresql+psycopg2://postgres:@localhost:5432/postgres'

print('=' * 60)
print('ETL DATA EXPLORER - CONFIGURATION')
print('=' * 60)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'DATA_DIR: {DATA_DIR}')
print(f'Python: {sys.version.split()[0]}')
CFG.display_summary()


ETL DATA EXPLORER - CONFIGURATION
PROJECT_ROOT: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform
DATA_DIR: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform\data
Python: 3.13.9
FEATURE FLAGS CONFIGURATION

Core Features:
  Financial Prediction:        ✓ Enabled
  Database Connection:         ✗ Disabled
  Advanced Analytics:          ✓ Enabled
  Dimensionality Reduction:    ✗ Disabled

Analysis Features:
  Sector Analysis:             ✓ Enabled
  Region Analysis:             ✓ Enabled

Output Features:
  Interactive Plots:           ✓ Enabled
  Excel Export:                ✓ Enabled
  Portfolio Optimization:      ✓ Enabled

Development:
  Debug Mode:                  ✗ Disabled


## Cell 2: Import Finance ML Modules

Import unified ETL pipeline, EDA analytics, and feature engineering modules.


In [2]:
# ============================================================================
# Cell 2: Import Finance ML Modules
# ============================================================================

# ETL Pipeline (Phase 9.1)
from finance_ml.ml_workflow.preprocessing.etl import (
    run_etl_pipeline,
    ETLConfig,
    ETLMetrics,
    etl_with_imputation,
    etl_from_csv,
    )

# EDA Analytics (Phase 9.2)
from finance_ml.ml_workflow.eda.eda import (
    eda_summary,
    generate_phase93_coverage_report,
    sector_distribution_summary,
    correlation_analysis,
    )

# Phase 9.3 Feature Categories
from finance_ml.ml_workflow.eda.phase93_categories import (
    PHASE93_FEATURE_CATEGORIES,
    categorize_dataframe_columns,
    get_phase93_coverage_stats,
    get_category_description,
    list_all_phase93_features,
    )

# Feature Engineering API (Phase 9.3)
from finance_ml.ml_workflow.features.api import build_features

# Column Semantics and Safety (Section 8.5)
from finance_ml.ml_workflow.preprocessing.column_semantics import (
    PRICE_COLUMNS,
    get_winsorizable_columns,
    get_scalable_columns,
    classify_columns,
    )

print('✓ Finance ML modules imported successfully')
print(f'✓ Phase 9.3 Feature Categories: {len(PHASE93_FEATURE_CATEGORIES)} categories')
print(f'✓ Total Phase 9.3 Features: {len(list_all_phase93_features())} features')
print(f'✓ Price Columns Protected: {len(PRICE_COLUMNS)} columns')
print(f'\nFeature Engineering Presets Available:')
print(f"  - 'basic': core ratios, margins, volatility, revenue CAGR")
print(f"  - 'momentum': momentum & technical indicators")
print(f"  - 'quality': accounting quality and financial distress signals")
print(f"  - 'comprehensive': full advanced feature set (196 features)")


✓ Finance ML modules imported successfully
✓ Phase 9.3 Feature Categories: 16 categories
✓ Total Phase 9.3 Features: 196 features
✓ Price Columns Protected: 21 columns

Feature Engineering Presets Available:
  - 'basic': core ratios, margins, volatility, revenue CAGR
  - 'momentum': momentum & technical indicators
  - 'quality': accounting quality and financial distress signals
  - 'comprehensive': full advanced feature set (196 features)


## Cell 3: Database Configuration

Configure PostgreSQL database connection with automatic URL format conversion.


In [3]:
# ============================================================================
# Cell 3: Database Configuration
# ============================================================================

# SQL file paths
SQL_SCHEMA = PROJECT_ROOT / 'create_equities_schema.sql'
SQL_IMPORT = PROJECT_ROOT / 'import_equities_data.sql'

# Detect availability
have_db_url = DB_URL is not None and len(DB_URL) > 0
CFG.have_database_connection = bool(have_db_url and HAVE_SQLALCHEMY)

print('=' * 60)
print('DATABASE CONFIGURATION')
print('=' * 60)
print(f'SQLAlchemy installed:  {HAVE_SQLALCHEMY}')
print(f'DB_URL configured:     {have_db_url}')
print(f'Database available:    {CFG.have_database_connection}')

if have_db_url:
    # Mask password for security
    try:
        parts = DB_URL.split('@')
        if len(parts) == 2:
            user_part = parts[0].split('//')[-1].split(':')[0]
            masked_url = f'postgresql://{user_part}@{parts[1]}'
        else:
            masked_url = 'postgresql://***@localhost:5432/postgres'
    except:
        masked_url = 'postgresql://***'
    print(f'Connection:            {masked_url}')
else:
    print('Connection:            Not configured (will use CSV fallback)')

print(f'\nSQL Files:')
print(f"  Schema script:       {'✓' if SQL_SCHEMA.exists() else '✗'} {SQL_SCHEMA.name}")
print(f"  Import script:       {'✓' if SQL_IMPORT.exists() else '✗'} {SQL_IMPORT.name}")
print('=' * 60)


DATABASE CONFIGURATION
SQLAlchemy installed:  True
DB_URL configured:     True
Database available:    True
Connection:            postgresql://postgres@localhost:5432/postgres

SQL Files:
  Schema script:       ✓ create_equities_schema.sql
  Import script:       ✓ import_equities_data.sql


## Cell 4: ETL Pipeline - Extract, Transform, Load

Run the unified ETL pipeline with 6-step imputation strategy.


In [4]:
# ============================================================================
# Cell 4: ETL Pipeline - Extract, Transform, Load
# ============================================================================

print('=' * 60)
print('ETL PIPELINE EXECUTION')
print('=' * 60)

# Configure ETL pipeline
etl_config = ETLConfig(
        normalize_columns=True,
        validate_schema=True,
        require_target=False,
        sanitize_data=True,
        apply_log_transforms=False,
        validate_quality=True,
        validate_pipeline=True,
        drop_invalid_rows=True,
        apply_imputation=True,
        imputation_strategy='6step',
        knn_neighbors=5,
        handle_categorical_imputation=True,
        handle_datetime_imputation=True,
        apply_scaling=False,
        )

# Run ETL pipeline (auto-detect source)
try:
    if CFG.have_database_connection:
        print('Attempting database connection...')
        all_stocks_preprocessed, etl_metrics = run_etl_pipeline(
                source='db',
                db_url=DB_URL,
                config=etl_config,
                return_metrics=True,
                )
    else:
        print('Using CSV data source...')
        all_stocks_preprocessed, etl_metrics = run_etl_pipeline(
                source='csv',
                data_dir=str(DATA_DIR),
                config=etl_config,
                return_metrics=True,
                )
except Exception as e:
    print(f'⚠ Primary source failed: {e}')
    print('Falling back to CSV...')
    all_stocks_preprocessed, etl_metrics = run_etl_pipeline(
            source='csv',
            data_dir=str(DATA_DIR),
            config=etl_config,
            return_metrics=True,
            )

# Display ETL metrics summary
print('\n' + '=' * 60)
print(etl_metrics.summary())
print('=' * 60)

# Validation checkpoint
assert not all_stocks_preprocessed.empty, 'Preprocessed data must not be empty'
assert 'ticker' in all_stocks_preprocessed.columns, 'ticker column must be present'
assert 'sector' in all_stocks_preprocessed.columns, 'sector column must be present'

print(f'\n✓ ETL Pipeline Complete')
print(f'  Data shape: {all_stocks_preprocessed.shape}')
print(f'  Source: {etl_metrics.source_type}')
print(f'  Duration: {etl_metrics.total_time_sec:.2f}s')
print(f'  Quality score: {etl_metrics.quality_score:.3f}')


ERROR:finance_ml.ml_workflow.preprocessing.etl:ETL pipeline failed: 'function' object has no attribute 'cursor'


ETL PIPELINE EXECUTION
Attempting database connection...
⚠ Primary source failed: 'function' object has no attribute 'cursor'
Falling back to CSV...

ETL Pipeline Summary:
  Source: csv
  Duration: 0.51s (extract: 0.19s, transform: 0.27s, load: 0.05s)
  Data: 6990 → 6989 rows, 350 → 350 columns
  Imputation: 6step (13413 → 0 missing) ✓
  Quality: 1.000, Validation: 0.857
  Stages: extract, transform, load
  Warnings: 0, Errors: 0

✓ ETL Pipeline Complete
  Data shape: (6989, 350)
  Source: csv
  Duration: 0.51s
  Quality score: 1.000


## Cell 5: Data Quality Overview

Examine data shape, dtypes, and missing values post-imputation.


In [5]:
# ============================================================================
# Cell 5: Data Quality Overview
# ============================================================================

print('=' * 60)
print('DATA QUALITY OVERVIEW')
print('=' * 60)

# Basic info
print(f'\nDataFrame Shape: {all_stocks_preprocessed.shape[0]:,} rows × {all_stocks_preprocessed.shape[1]} columns')
print(f'Memory Usage: {all_stocks_preprocessed.memory_usage(deep=True).sum() / 1024 ** 2:.2f} MB')

# Data types summary
dtype_counts = all_stocks_preprocessed.dtypes.value_counts()
print(f'\nData Types:')
for dtype, count in dtype_counts.items():
    print(f'  {dtype}: {count} columns')

# Missing values (post-imputation)
missing = all_stocks_preprocessed.isnull().sum()
missing_pct = (missing / len(all_stocks_preprocessed) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing': missing.values,
    'Percent': missing_pct.values
    }).query('Missing > 0').sort_values('Missing', ascending=False)

if len(missing_df) > 0:
    print(f'\n⚠ Columns with missing values (Top 10):')
    print(missing_df.head(10).to_string(index=False))
else:
    print(f'\n✓ No missing values - imputation successful!')

# Summary statistics for key columns
print(f'\nSummary Statistics (key numeric columns):')
key_cols = ['last_price', 'market_cap', 'enterprise_value', 'ebitda_ltm', 'p_e_ntm']
available_keys = [c for c in key_cols if c in all_stocks_preprocessed.columns]
if available_keys:
    print(all_stocks_preprocessed[available_keys].describe().round(2).to_string())


DATA QUALITY OVERVIEW

DataFrame Shape: 6,989 rows × 350 columns
Memory Usage: 32.03 MB

Data Types:
  float64: 327 columns
  object: 16 columns
  datetime64[ns]: 7 columns

✓ No missing values - imputation successful!

Summary Statistics (key numeric columns):
       last_price  market_cap  enterprise_value  ebitda_ltm  p_e_ntm
count     6989.00     6989.00           6989.00     6989.00  6989.00
mean      3268.64    10041.75          10635.92      785.23    22.02
std      36780.40    82399.35          82670.58     4495.98    29.93
min          0.00        2.78          -4682.22    -2554.31     0.00
25%          7.65      499.78            723.97       33.21    11.20
50%         26.40     1660.21           1909.10      132.06    15.10
75%        102.76     4937.87           5279.05      411.62    21.20
max    1901000.00  4300923.00        4251137.00   217535.33   494.30


## Cell 6: Phase 9.3 Feature Engineering

**Business Goal:** Engineer comprehensive financial features including valuation ratios, profitability metrics, quality indicators, and sector-specific features to maximize model predictive power.

**Key Objectives:**
1. Engineer valuation ratios (P/E, P/B, EV/EBITDA, PEG)
2. Engineer profitability features (margins, ROE, ROA, ROIC)
3. Create momentum and technical indicators
4. Engineer analyst quality features
5. Create accounting quality scores (Altman Z, Piotroski F)
6. Build sector-relative features
7. Create interaction features


In [6]:
# ============================================================================
# Cell 6: Phase 9.3 Feature Engineering
# ============================================================================

import time

feature_eng_start = time.time()

print('=' * 80)
print('PHASE 9.3: ADVANCED FEATURE ENGINEERING')
print('=' * 80)

# Build comprehensive features using Phase 9.3 API
print('\n📊 Building comprehensive feature set (196 features)...')
print(f'  Input DataFrame shape: {all_stocks_preprocessed.shape}')

all_stocks_features = build_features(
        all_stocks_preprocessed,
        preset='comprehensive',
        include_interactions=True,
        include_relative=True,
        sector_col='sector'
        )

feature_eng_time = time.time() - feature_eng_start

print(f'\n✓ Feature Engineering Complete')
print(f'  Original columns:     {all_stocks_preprocessed.shape[1]}')
print(f'  Engineered columns:   {all_stocks_features.shape[1]}')
print(f'  New features added:   {all_stocks_features.shape[1] - all_stocks_preprocessed.shape[1]}')
print(f'  Duration:             {feature_eng_time:.2f}s')

# Validation checkpoint
assert not all_stocks_features.empty, 'Feature engineered data must not be empty'
assert all_stocks_features.shape[1] > all_stocks_preprocessed.shape[1], 'Features must be added'

# Phase 9.3 Enhanced Benchmarking Analysis
print('\n' + '=' * 80)
print('📊 Phase 9.3 Enhanced Benchmarking Analysis:')
print('=' * 80)

# Categorize features by Phase 9.3 families
categorized = categorize_dataframe_columns(all_stocks_features)
coverage_stats = get_phase93_coverage_stats(all_stocks_features)

# Calculate total Phase 9.3 features present
total_phase93_features = sum(coverage_stats.values())
expected_counts = {cat: len(features) for cat, features in PHASE93_FEATURE_CATEGORIES.items()}
total_expected = sum(expected_counts.values())

print(f'\n✓ Analyzing engineered features DataFrame')
print(f'  Total stocks: {all_stocks_features.shape[0]}')
print(f'  Total columns: {all_stocks_features.shape[1]}')
print(
    f'  Phase 9.3 engineered features present: {total_phase93_features}/{total_expected} ({total_phase93_features / total_expected * 100:.1f}%)')

if 'sector' in all_stocks_features.columns:
    print(f'  Sectors analyzed: {all_stocks_features["sector"].nunique()}')
if 'region' in all_stocks_features.columns:
    print(f'  Regions analyzed: {all_stocks_features["region"].nunique()}')

# Show coverage by category
print(f'\n📋 Phase 9.3 Feature Coverage by Category:')
print('=' * 80)

for category in sorted(PHASE93_FEATURE_CATEGORIES.keys()):
    present = coverage_stats.get(category, 0)
    expected = expected_counts[category]

    if present > 0:
        pct = (present / expected * 100) if expected > 0 else 0
        print(f'  ✓ {category}: {present}/{expected} features ({pct:.1f}% coverage)')

        # Show sample features for this category
        if category in categorized:
            sample_features = categorized[category][:3]
            for feat in sample_features:
                non_null = all_stocks_features[feat].notna().sum()
                print(f'      • {feat}: {non_null}/{len(all_stocks_features)} non-null')
    else:
        print(f'  ✗ {category}: 0/{expected} features (not yet engineered)')

# Summary
print(f'\n📊 Benchmarking Summary:')
print('=' * 80)
categories_with_features = len([c for c in coverage_stats.values() if c > 0])
print(f'  Categories with features: {categories_with_features}/{len(PHASE93_FEATURE_CATEGORIES)}')
print(f'  Total Phase 9.3 features: {total_phase93_features}')
print(f'  Overall coverage: {total_phase93_features / total_expected * 100:.1f}%')

# Export benchmarking report
benchmarking_report_path = OUTPUT_DIR / 'eda' / 'phase93_benchmarking_etl_explorer.json'
benchmarking_summary = {
    'phase': '9.3',
    'data_source': 'all_stocks_features DataFrame (ETL Data Explorer)',
    'timestamp': pd.Timestamp.now().isoformat(),
    'total_stocks': int(all_stocks_features.shape[0]),
    'total_columns': int(all_stocks_features.shape[1]),
    'phase93_features_present': int(total_phase93_features),
    'phase93_features_expected': int(total_expected),
    'coverage_percentage': float(total_phase93_features / total_expected * 100),
    'category_coverage': {
        cat: {
            'present': int(coverage_stats.get(cat, 0)),
            'expected': int(expected_counts[cat]),
            'coverage_pct': float(
                (coverage_stats.get(cat, 0) / expected_counts[cat] * 100) if expected_counts[cat] > 0 else 0)
            }
        for cat in PHASE93_FEATURE_CATEGORIES.keys()
        },
    'categories_with_features': categories_with_features,
    'feature_engineering_duration_sec': feature_eng_time,
    }

with open(benchmarking_report_path, 'w') as f:
    json.dump(benchmarking_summary, f, indent=2)
print(f'\n✓ Benchmarking report saved to: {benchmarking_report_path}')


PHASE 9.3: ADVANCED FEATURE ENGINEERING

📊 Building comprehensive feature set (196 features)...
  Input DataFrame shape: (6989, 350)

✓ Feature Engineering Complete
  Original columns:     350
  Engineered columns:   572
  New features added:   222
  Duration:             0.73s

📊 Phase 9.3 Enhanced Benchmarking Analysis:

✓ Analyzing engineered features DataFrame
  Total stocks: 6989
  Total columns: 572
  Phase 9.3 engineered features present: 182/196 (92.9%)
  Sectors analyzed: 11
  Regions analyzed: 5

📋 Phase 9.3 Feature Coverage by Category:
  ✓ Analyst Sentiment: 10/10 features (100.0% coverage)
      • price_target_spread_pct: 6987/6989 non-null
      • price_target_range: 6987/6989 non-null
      • consensus_strength: 6987/6989 non-null
  ✓ Balance Sheet Dynamics: 8/8 features (100.0% coverage)
      • debt_growth_rate: 6807/6989 non-null
      • equity_growth_rate: 6989/6989 non-null
      • asset_growth_rate: 6989/6989 non-null
  ✓ Capital Allocation: 21/23 features (91.3% c

## Cell 7: Region & Sector Analytics with Feature Coverage

Section 17 compliant Plotly visualizations with dark theme.
Enhanced with Phase 9.3 feature analytics by region and sector.


In [7]:
# ============================================================================
# Cell 7: Region & Sector Analytics with Feature Coverage
# ============================================================================

print('=' * 80)
print('REGION & SECTOR ANALYTICS WITH FEATURE COVERAGE')
print('=' * 80)

# Use all_stocks_features (post feature engineering) for analytics
df_analytics = all_stocks_features

# Region distribution
if 'region' in df_analytics.columns:
    region_counts = df_analytics['region'].value_counts().reset_index()
    region_counts.columns = ['Region', 'Count']
    region_counts['Percentage'] = (region_counts['Count'] / region_counts['Count'].sum() * 100).round(1)

    fig_region = px.bar(
            region_counts,
            x='Count',
            y='Region',
            orientation='h',
            title='Stock Distribution by Region (Post Feature Engineering)',
            template=PLOTLY_TEMPLATE,
            color='Count',
            color_continuous_scale='Blues',
            text='Count',
            )
    fig_region.update_traces(
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Count: %{x}<br>Percentage: %{customdata[0]:.1f}%',
            customdata=region_counts[['Percentage']].values
            )
    fig_region.update_layout(
            xaxis_title='Number of Stocks',
            yaxis_title='Region',
            height=400,
            showlegend=False,
            )
    fig_region.show()

# Sector distribution (Top 15)
if 'sector' in df_analytics.columns:
    sector_counts = df_analytics['sector'].value_counts().head(15).reset_index()
    sector_counts.columns = ['Sector', 'Count']
    sector_counts['Percentage'] = (sector_counts['Count'] / len(df_analytics) * 100).round(1)

    fig_sector = px.bar(
            sector_counts.sort_values('Count'),
            x='Count',
            y='Sector',
            orientation='h',
            title='Stock Distribution by Sector (Top 15, Post Feature Engineering)',
            template=PLOTLY_TEMPLATE,
            color='Count',
            color_continuous_scale='Greens',
            text='Count',
            )
    fig_sector.update_traces(
            textposition='outside',
            hovertemplate='<b>%{y}</b><br>Count: %{x}<br>Percentage: %{customdata[0]:.1f}%',
            customdata=sector_counts.sort_values('Count')[['Percentage']].values
            )
    fig_sector.update_layout(
            xaxis_title='Number of Stocks',
            yaxis_title='Sector',
            height=500,
            showlegend=False,
            )
    fig_sector.show()

# Region-Sector Heatmap
if 'region' in df_analytics.columns and 'sector' in df_analytics.columns:
    cross_tab = pd.crosstab(
            df_analytics['sector'],
            df_analytics['region']
            ).head(12)

    fig_heatmap = px.imshow(
            cross_tab,
            title='Region-Sector Distribution Heatmap',
            template=PLOTLY_TEMPLATE,
            color_continuous_scale='Viridis',
            aspect='auto',
            )
    fig_heatmap.update_layout(
            xaxis_title='Region',
            yaxis_title='Sector',
            height=500,
            )
    fig_heatmap.show()

print('\n✓ Distribution visualizations complete')

# ============================================================================
# Feature Coverage Analytics by Region and Sector
# ============================================================================
print('\n' + '=' * 80)
print('📊 PHASE 9.3 FEATURE ANALYTICS BY REGION & SECTOR')
print('=' * 80)

# Define key Phase 9.3 features for analysis
key_phase93_features = [
    'roe', 'roa', 'roic',  # Profitability
    'p_e_ratio', 'p_s_ratio', 'ev_ebitda_ratio',  # Valuation
    'price_momentum_1m', 'price_momentum_3m', 'price_momentum_6m',  # Momentum
    'debt_to_equity', 'interest_coverage',  # Leverage
    'piotroski_f_score', 'altman_z_score',  # Composite Scores
    ]
available_phase93 = [f for f in key_phase93_features if f in df_analytics.columns]

if available_phase93 and 'region' in df_analytics.columns:
    print(f'\n📍 Feature Coverage by Region ({len(available_phase93)} key features):')

    # Calculate non-null percentage per region for each feature
    region_coverage_data = []
    for region in df_analytics['region'].unique():
        region_df = df_analytics[df_analytics['region'] == region]
        region_count = len(region_df)

        for feat in available_phase93:
            non_null = region_df[feat].notna().sum()
            coverage_pct = (non_null / region_count * 100) if region_count > 0 else 0
            region_coverage_data.append({
                'Region': region,
                'Feature': feat,
                'Coverage %': round(coverage_pct, 1),
                'Non-Null': non_null,
                'Total': region_count
                })

    region_coverage_df = pd.DataFrame(region_coverage_data)

    # Pivot for heatmap
    region_pivot = region_coverage_df.pivot(index='Feature', columns='Region', values='Coverage %')

    fig_region_coverage = px.imshow(
            region_pivot,
            title='Phase 9.3 Feature Coverage by Region (%)',
            template=PLOTLY_TEMPLATE,
            color_continuous_scale='RdYlGn',
            zmin=0, zmax=100,
            aspect='auto',
            )
    fig_region_coverage.update_layout(
            xaxis_title='Region',
            yaxis_title='Feature',
            height=500,
            )
    fig_region_coverage.show()

if available_phase93 and 'sector' in df_analytics.columns:
    print(f'\n🏢 Feature Coverage by Sector (Top 10 sectors, {len(available_phase93)} key features):')

    # Get top 10 sectors by count
    top_sectors = df_analytics['sector'].value_counts().head(10).index.tolist()

    # Calculate non-null percentage per sector for each feature
    sector_coverage_data = []
    for sector in top_sectors:
        sector_df = df_analytics[df_analytics['sector'] == sector]
        sector_count = len(sector_df)

        for feat in available_phase93:
            non_null = sector_df[feat].notna().sum()
            coverage_pct = (non_null / sector_count * 100) if sector_count > 0 else 0
            sector_coverage_data.append({
                'Sector': str(sector)[:25],  # Truncate long names
                'Feature': feat,
                'Coverage %': round(coverage_pct, 1),
                'Non-Null': non_null,
                'Total': sector_count
                })

    sector_coverage_df = pd.DataFrame(sector_coverage_data)

    # Pivot for heatmap
    sector_pivot = sector_coverage_df.pivot(index='Feature', columns='Sector', values='Coverage %')

    fig_sector_coverage = px.imshow(
            sector_pivot,
            title='Phase 9.3 Feature Coverage by Sector (%)',
            template=PLOTLY_TEMPLATE,
            color_continuous_scale='RdYlGn',
            zmin=0, zmax=100,
            aspect='auto',
            )
    fig_sector_coverage.update_layout(
            xaxis_title='Sector',
            yaxis_title='Feature',
            height=600,
            )
    fig_sector_coverage.show()

# Feature statistics by region
if available_phase93 and 'region' in df_analytics.columns:
    print(f'\n📊 Key Feature Statistics by Region:')

    # Select 3 key features for detailed statistics
    stat_features = ['roe', 'price_momentum_1m', 'debt_to_equity']
    stat_features = [f for f in stat_features if f in df_analytics.columns]

    if stat_features:
        region_stats = df_analytics.groupby('region')[stat_features].agg(['mean', 'median', 'std']).round(3)
        print(region_stats.to_string())

        # Box plot for ROE by region (if available)
        if 'roe' in df_analytics.columns:
            fig_roe_region = px.box(
                    df_analytics[df_analytics['roe'].notna()],
                    x='region',
                    y='roe',
                    title='Return on Equity (ROE) Distribution by Region',
                    template=PLOTLY_TEMPLATE,
                    color='region',
                    )
            fig_roe_region.update_layout(
                    xaxis_title='Region',
                    yaxis_title='ROE',
                    height=450,
                    showlegend=False,
                    )
            fig_roe_region.show()

print('\n✓ Feature analytics by region and sector complete')


REGION & SECTOR ANALYTICS WITH FEATURE COVERAGE



✓ Distribution visualizations complete

📊 PHASE 9.3 FEATURE ANALYTICS BY REGION & SECTOR

📍 Feature Coverage by Region (13 key features):



🏢 Feature Coverage by Sector (Top 10 sectors, 13 key features):



📊 Key Feature Statistics by Region:
                                roe                  price_momentum_1m                  debt_to_equity               
                               mean  median      std              mean median       std           mean median     std
region                                                                                                               
Africa / Middle East         24.293  13.439  318.145             7.011 -2.588   245.594          0.784  0.542   2.113
Asia / Pacific                9.096   8.743   22.491             4.636 -1.003   132.436          0.597  0.313   2.038
Europe                      -16.266   8.843  991.910            66.514 -1.359  3080.259          0.524  0.544  18.242
Latin America and Caribbean   8.162   9.748   53.413             6.759  4.460    20.139          1.400  0.786   2.412
United States and Canada    -35.030   6.047  901.356             0.487  1.012    18.541          0.373  0.375  27.697



✓ Feature analytics by region and sector complete


## Cell 8: Numeric Feature Distributions

Examine distributions of key numeric metrics with statistical annotations (using engineered features).


In [8]:
# ============================================================================
# Cell 8: Numeric Feature Distributions
# ============================================================================

# Key metrics for comprehensive feature store (includes Phase 9.3 engineered features)
key_metrics = [
    'last_price', 'market_cap', 'enterprise_value', 'ebitda_ltm', 'p_e_ntm', 'beta_5y',
    'roe', 'roa', 'price_momentum_1m', 'piotroski_f_score'  # Phase 9.3 features
    ]
available_metrics = [m for m in key_metrics if m in all_stocks_features.columns]

if available_metrics:
    n_metrics = len(available_metrics)
    n_cols = 3
    n_rows = math.ceil(n_metrics / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.ravel() if n_metrics > 1 else [axes]

    for i, metric in enumerate(available_metrics):
        data = all_stocks_features[metric].dropna()

        if len(data) > 0:
            ax = axes[i]
            sns.histplot(data, bins=50, kde=True, ax=ax, color=COLOR_PALETTE['primary'])

            # Add statistics
            mean_val = data.mean()
            median_val = data.median()

            ax.axvline(mean_val, color=COLOR_PALETTE['warning'], linestyle='--',
                       label=f'Mean: {mean_val:,.2f}', linewidth=2)
            ax.axvline(median_val, color=COLOR_PALETTE['success'], linestyle='--',
                       label=f'Median: {median_val:,.2f}', linewidth=2)

            ax.set_title(f'Distribution: {metric}', fontsize=14, fontweight='bold')
            ax.set_xlabel(metric.replace('_', ' ').title())
            ax.set_ylabel('Frequency')
            ax.legend()
            ax.grid(True, alpha=0.3)

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

    print(f'✓ Visualized {len(available_metrics)} key metrics')
else:
    print('⚠ No key metrics found in dataset')

# Correlation heatmap for key metrics (using engineered features)
numeric_cols = all_stocks_features.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) >= 4:
    # Select top correlated features (prioritize Phase 9.3 features)
    corr_subset = numeric_cols[:15]  # Limit for readability
    corr_matrix = all_stocks_features[corr_subset].corr()

    fig_corr = px.imshow(
            corr_matrix,
            title='Feature Correlation Heatmap',
            template=PLOTLY_TEMPLATE,
            color_continuous_scale='RdBu_r',
            zmin=-1, zmax=1,
            aspect='auto',
            )
    fig_corr.update_layout(height=600)
    fig_corr.show()


✓ Visualized 10 key metrics


## Cell 9: Phase 9.3 Feature Category Coverage

Analyze coverage of the 16 Phase 9.3 feature categories (196 features) after feature engineering.


In [9]:
# ============================================================================
# Cell 9: Phase 9.3 Feature Category Coverage
# ============================================================================

print('=' * 80)
print('PHASE 9.3 FEATURE CATEGORY COVERAGE (Post Feature Engineering)')
print('=' * 80)

# Generate comprehensive coverage report using engineered features
coverage_report = generate_phase93_coverage_report(all_stocks_features)

# Build coverage DataFrame
coverage_data = []
for cat, features in PHASE93_FEATURE_CATEGORIES.items():
    present_count = coverage_report.get('category_breakdown', {}).get(cat, 0)
    total_count = len(features)
    coverage_pct = (present_count / total_count * 100) if total_count > 0 else 0
    coverage_data.append({
        'Category': cat,
        'Present': present_count,
        'Total': total_count,
        'Coverage %': round(coverage_pct, 1),
        'Description': get_category_description(cat)[:40] + '...'
        })

coverage_df = pd.DataFrame(coverage_data).sort_values('Present', ascending=False)

print(
    f"\nTotal Phase 9.3 Features Present: {coverage_report.get('total_phase93_features', 0)} / {len(list_all_phase93_features())}")
print(f"Overall Coverage: {coverage_report.get('coverage_percentage', 0):.1f}%\n")
print(coverage_df.to_string(index=False))

# Treemap visualization
treemap_data = coverage_df[coverage_df['Present'] > 0].copy()
if not treemap_data.empty:
    fig_treemap = px.treemap(
            treemap_data,
            path=['Category'],
            values='Present',
            title='Phase 9.3 Feature Coverage by Category',
            template=PLOTLY_TEMPLATE,
            color='Coverage %',
            color_continuous_scale='Viridis',
            hover_data=['Description', 'Total']
            )
    fig_treemap.update_traces(
            textinfo='label+value',
            hovertemplate='<b>%{label}</b><br>Present: %{value}<br>Coverage: %{color:.1f}%<extra></extra>'
            )
    fig_treemap.update_layout(height=600)
    fig_treemap.show()

# Bar chart for coverage comparison
fig_coverage = px.bar(
        coverage_df.sort_values('Coverage %', ascending=True),
        x='Coverage %',
        y='Category',
        orientation='h',
        title='Phase 9.3 Feature Coverage by Category',
        template=PLOTLY_TEMPLATE,
        color='Coverage %',
        color_continuous_scale='RdYlGn',
        text='Present'
        )
fig_coverage.update_traces(texttemplate='%{text}', textposition='outside')
fig_coverage.update_layout(
        xaxis_title='Coverage Percentage',
        yaxis_title='Feature Category',
        height=600,
        showlegend=False,
        )
fig_coverage.show()

print('\n✓ Phase 9.3 coverage analysis complete')


PHASE 9.3 FEATURE CATEGORY COVERAGE (Post Feature Engineering)

Total Phase 9.3 Features Present: 182 / 196
Overall Coverage: 92.9%

              Category  Present  Total  Coverage %                                 Description
  Momentum & Technical       25     27        92.6 Price momentum, technical indicators, EM...
      Valuation Ratios       21     23        91.3 EV/EBITDA, P/E, P/B, and other valuation...
    Capital Allocation       21     23        91.3 Dividends, buybacks, reinvestment, M&A i...
        Quality & Risk       18     18       100.0 Accounting quality, earnings quality, di...
 Employee Productivity       16     16       100.0 Workforce metrics, employee growth, reve...
     Temporal Patterns       13     16        81.2 Seasonality, trend consistency, cyclical...
         Profitability       12     12       100.0 ROE, ROA, profit margins, and profitabil...
     Analyst Sentiment       10     10       100.0 Analyst ratings, consensus strength, tar...
   Revenue F


✓ Phase 9.3 coverage analysis complete


## Cell 10: Financial Metrics Analytics

Comprehensive financial analytics with statistical summaries, hypothesis testing, and interactive visualizations.

**Key Objectives:**
1. Generate comprehensive statistical summaries and financial analytics reports
2. Analyze correlations and multicollinearity between features
3. Perform hypothesis testing across features, sectors, industry, country and regions
4. Create interactive visualizations for financial metrics distributions and relationships
5. Generate benchmarking reports comparing sector and regional financial/market performance

**Outputs:**
- **JSON Reports (4 files):** eda_summary.json, data_quality_alerts.json, metrics_dashboard.json, hypothesis_tests.json
- **Interactive Visualizations (7 HTML files):** correlation_heatmap.html, distributions.html, valuation_3d.html, region_sector_heatmap.html, sector_boxplots.html, regional_comparison.html, phase93_category_sector_bubble_chart.html


In [10]:
# ============================================================================
# Cell 10: Financial Metrics Analytics
# ============================================================================

import scipy.stats as stats
from datetime import datetime

# Import analytics functions
from finance_ml.ml_workflow.analytics.eval import (
    calculate_financial_metrics_dashboard,
    generate_data_quality_alerts,
    perform_comprehensive_hypothesis_tests,
    calculate_correlation_matrix,
    find_top_correlations,
    create_region_sector_heatmap,
    )
from finance_ml.ml_workflow.eda.eda import eda_summary
from finance_ml.ml_workflow.eda.reports import generate_benchmarking_report

print('=' * 80)
print('FINANCIAL METRICS ANALYTICS')
print('=' * 80)

# Create output directories
financial_metrics_dir = OUTPUT_DIR / 'eda' / 'financial_metrics'
financial_metrics_dir.mkdir(parents=True, exist_ok=True)

# ============================================================================
# 1. Generate JSON Reports
# ============================================================================
print('\n📊 Generating JSON Reports...')

# 1.1 EDA Summary Report
print('  → eda_summary.json')
eda_summary_data = eda_summary(all_stocks_features, sector_column='sector', include_correlations=True)
eda_summary_path = financial_metrics_dir / 'eda_summary.json'
with open(eda_summary_path, 'w') as f:
    # Convert non-serializable objects
    def make_serializable(obj):
        if isinstance(obj, (np.integer, np.floating)):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, pd.Timestamp):
            return obj.isoformat()
        elif isinstance(obj, dict):
            return {k: make_serializable(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [make_serializable(i) for i in obj]
        return obj


    json.dump(make_serializable(eda_summary_data), f, indent=2, default=str)
print(f'    ✓ Saved: {eda_summary_path}')

# 1.2 Data Quality Alerts Report
print('  → data_quality_alerts.json')
quality_alerts = generate_data_quality_alerts(all_stocks_features, outlier_threshold=3.0)
quality_alerts_path = financial_metrics_dir / 'data_quality_alerts.json'
with open(quality_alerts_path, 'w') as f:
    json.dump(make_serializable(quality_alerts), f, indent=2, default=str)
print(f'    ✓ Saved: {quality_alerts_path}')

# 1.3 Financial Metrics Dashboard
print('  → metrics_dashboard.json')
# By sector
metrics_by_sector = calculate_financial_metrics_dashboard(all_stocks_features, group_by='sector')
# By region
metrics_by_region = calculate_financial_metrics_dashboard(all_stocks_features, group_by='region')
metrics_dashboard = {
    'timestamp': datetime.now().isoformat(),
    'total_stocks': len(all_stocks_features),
    'by_sector': make_serializable(metrics_by_sector),
    'by_region': make_serializable(metrics_by_region),
    }
metrics_dashboard_path = financial_metrics_dir / 'metrics_dashboard.json'
with open(metrics_dashboard_path, 'w') as f:
    json.dump(metrics_dashboard, f, indent=2, default=str)
print(f'    ✓ Saved: {metrics_dashboard_path}')

# 1.4 Hypothesis Tests Report
print('  → hypothesis_tests.json')
test_metrics = ['roe', 'roa', 'p_e_ratio', 'debt_to_equity', 'price_momentum_1m']
test_metrics = [m for m in test_metrics if m in all_stocks_features.columns]
hypothesis_results = perform_comprehensive_hypothesis_tests(
        all_stocks_features,
        group_column='sector',
        metrics=test_metrics,
        alpha=0.05
        )
hypothesis_path = financial_metrics_dir / 'hypothesis_tests.json'
with open(hypothesis_path, 'w') as f:
    json.dump(make_serializable(hypothesis_results), f, indent=2, default=str)
print(f'    ✓ Saved: {hypothesis_path}')

print(f'\n✓ JSON Reports Complete: 4 files generated')

# ============================================================================
# 2. Generate Interactive HTML Visualizations
# ============================================================================
print('\n📈 Generating Interactive HTML Visualizations...')

# 2.1 Correlation Heatmap (Top 50 features)
print('  → correlation_heatmap.html')
numeric_features = all_stocks_features.select_dtypes(include=[np.number]).columns.tolist()
# Select top 50 most complete numeric features
feature_completeness = all_stocks_features[numeric_features].notna().sum().sort_values(ascending=False)
top_50_features = feature_completeness.head(50).index.tolist()

corr_matrix = all_stocks_features[top_50_features].corr()

# Cluster the correlation matrix for better visualization
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform

# Handle NaN in correlation matrix
corr_matrix_filled = corr_matrix.fillna(0)
try:
    # Compute distance matrix and linkage
    dist_matrix = 1 - np.abs(corr_matrix_filled)
    np.fill_diagonal(dist_matrix.values, 0)
    linkage = hierarchy.linkage(squareform(dist_matrix), method='average')
    order = hierarchy.leaves_list(linkage)
    corr_clustered = corr_matrix_filled.iloc[order, order]
except:
    corr_clustered = corr_matrix_filled

fig_corr = px.imshow(
        corr_clustered,
        title='<b>Feature Correlation Heatmap</b><br><sup>Top 50 Features (Clustered)</sup>',
        template=PLOTLY_TEMPLATE,
        color_continuous_scale='RdBu_r',
        zmin=-1, zmax=1,
        aspect='auto',
        )
fig_corr.update_layout(
        height=900,
        width=1000,
        font=dict(family='Segoe UI, Roboto, Arial', size=10),
        title_font_size=20,
        xaxis_title='Features',
        yaxis_title='Features',
        )
fig_corr.write_html(financial_metrics_dir / 'correlation_heatmap.html')
print(f'    ✓ Saved: correlation_heatmap.html')

# 2.2 Feature Distributions by Sector
print('  → distributions.html')
key_distribution_features = ['roe', 'roa', 'p_e_ratio', 'debt_to_equity', 'piotroski_f_score', 'altman_z_score']
key_distribution_features = [f for f in key_distribution_features if f in all_stocks_features.columns]

if key_distribution_features and 'sector' in all_stocks_features.columns:
    # Create subplots for distributions
    fig_dist = make_subplots(
            rows=2, cols=3,
            subplot_titles=[f.replace('_', ' ').title() for f in key_distribution_features[:6]],
            vertical_spacing=0.12,
            horizontal_spacing=0.08,
            )

    colors = [COLOR_PALETTE['primary'], COLOR_PALETTE['success'], COLOR_PALETTE['warning'],
              COLOR_PALETTE['danger'], COLOR_PALETTE['info'], COLOR_PALETTE['neutral']]

    for idx, feat in enumerate(key_distribution_features[:6]):
        row = idx // 3 + 1
        col = idx % 3 + 1

        data = all_stocks_features[feat].dropna()
        fig_dist.add_trace(
                go.Histogram(
                        x=data,
                        name=feat,
                        marker_color=colors[idx % len(colors)],
                        opacity=0.7,
                        nbinsx=50,
                        hovertemplate=f'<b>{feat}</b><br>Range: %{{x}}<br>Count: %{{y}}<extra></extra>'
                        ),
                row=row, col=col
                )
        fig_dist.update_xaxes(title_text=feat.replace('_', ' ').title(), row=row, col=col)
        fig_dist.update_yaxes(title_text='Frequency', row=row, col=col)

    fig_dist.update_layout(
            title='<b>Financial Metrics Distributions</b><br><sup>Key Features Across All Stocks</sup>',
            template=PLOTLY_TEMPLATE,
            height=700,
            showlegend=False,
            font=dict(family='Segoe UI, Roboto, Arial'),
            title_font_size=20,
            )
    fig_dist.write_html(financial_metrics_dir / 'distributions.html')
    print(f'    ✓ Saved: distributions.html')

# 2.3 3D Valuation Scatter (Category-Sector-Market Cap)
print('  → valuation_3d.html')
# Select features for 3D visualization
val_features = {
    'x': 'roe' if 'roe' in all_stocks_features.columns else 'roa',
    'y': 'piotroski_f_score' if 'piotroski_f_score' in all_stocks_features.columns else 'altman_z_score',
    'z': 'market_cap' if 'market_cap' in all_stocks_features.columns else 'enterprise_value',
    }

if all(f in all_stocks_features.columns or f is None for f in val_features.values()):
    # Prepare data for 3D scatter
    df_3d = all_stocks_features[['ticker', 'sector', 'region'] + list(val_features.values())].dropna()

    # Log transform market cap for better visualization
    if 'market_cap' in df_3d.columns:
        df_3d['market_cap_log'] = np.log10(df_3d['market_cap'].clip(lower=1))
        z_col = 'market_cap_log'
        z_label = 'Market Cap (Log10 $)'
    else:
        z_col = val_features['z']
        z_label = val_features['z'].replace('_', ' ').title()

    fig_3d = px.scatter_3d(
            df_3d.head(2000),  # Limit for performance
            x=val_features['x'],
            y=val_features['y'],
            z=z_col,
            color='sector',
            symbol='region',
            hover_data=['ticker'],
            title='<b>Value vs Quality vs Size Trade-offs</b><br><sup>3D Category-Sector-Market Cap Analysis</sup>',
            template=PLOTLY_TEMPLATE,
            opacity=0.7,
            )
    fig_3d.update_layout(
            height=800,
            font=dict(family='Segoe UI, Roboto, Arial'),
            title_font_size=20,
            scene=dict(
                    xaxis_title=val_features['x'].replace('_', ' ').upper(),
                    yaxis_title=val_features['y'].replace('_', ' ').title(),
                    zaxis_title=z_label,
                    ),
            )
    fig_3d.write_html(financial_metrics_dir / 'valuation_3d.html')
    print(f'    ✓ Saved: valuation_3d.html')

# 2.4 Region-Sector Heatmap
print('  → region_sector_heatmap.html')
if 'region' in all_stocks_features.columns and 'sector' in all_stocks_features.columns:
    # Create pivot table for region-sector distribution
    region_sector_pivot = pd.crosstab(
            all_stocks_features['sector'],
            all_stocks_features['region'],
            margins=True
            )

    # Remove margins for heatmap
    region_sector_data = region_sector_pivot.iloc[:-1, :-1]

    fig_rs_heatmap = px.imshow(
            region_sector_data,
            title='<b>Regional Financial Analytics Distribution</b><br><sup>Stock Count by Sector and Region</sup>',
            template=PLOTLY_TEMPLATE,
            color_continuous_scale='Blues',
            aspect='auto',
            text_auto=True,
            )
    fig_rs_heatmap.update_layout(
            height=700,
            font=dict(family='Segoe UI, Roboto, Arial'),
            title_font_size=20,
            xaxis_title='Region',
            yaxis_title='Sector',
            )
    fig_rs_heatmap.write_html(financial_metrics_dir / 'region_sector_heatmap.html')
    print(f'    ✓ Saved: region_sector_heatmap.html')

# 2.5 Sector Boxplots
print('  → sector_boxplots.html')
boxplot_metrics = ['roe', 'roa', 'debt_to_equity', 'price_momentum_1m']
boxplot_metrics = [m for m in boxplot_metrics if m in all_stocks_features.columns]

if boxplot_metrics and 'sector' in all_stocks_features.columns:
    fig_box = make_subplots(
            rows=2, cols=2,
            subplot_titles=[m.replace('_', ' ').title() for m in boxplot_metrics[:4]],
            vertical_spacing=0.15,
            horizontal_spacing=0.1,
            )

    for idx, metric in enumerate(boxplot_metrics[:4]):
        row = idx // 2 + 1
        col = idx % 2 + 1

        # Get top 10 sectors by count
        top_sectors = all_stocks_features['sector'].value_counts().head(10).index.tolist()
        df_filtered = all_stocks_features[all_stocks_features['sector'].isin(top_sectors)]

        for i, sector in enumerate(top_sectors):
            sector_data = df_filtered[df_filtered['sector'] == sector][metric].dropna()
            fig_box.add_trace(
                    go.Box(
                            y=sector_data,
                            name=str(sector)[:15],
                            marker_color=px.colors.qualitative.Set3[i % 12],
                            showlegend=(idx == 0),
                            hovertemplate=f'<b>{sector}</b><br>{metric}: %{{y:.2f}}<extra></extra>'
                            ),
                    row=row, col=col
                    )
        fig_box.update_yaxes(title_text=metric.replace('_', ' ').title(), row=row, col=col)

    fig_box.update_layout(
            title='<b>Financial Metrics by Sector</b><br><sup>Box Plots for Key Metrics (Top 10 Sectors)</sup>',
            template=PLOTLY_TEMPLATE,
            height=800,
            font=dict(family='Segoe UI, Roboto, Arial'),
            title_font_size=20,
            showlegend=True,
            legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.5),
            )
    fig_box.write_html(financial_metrics_dir / 'sector_boxplots.html')
    print(f'    ✓ Saved: sector_boxplots.html')

# 2.6 Regional Comparison
print('  → regional_comparison.html')
if 'region' in all_stocks_features.columns:
    comparison_metrics = ['roe', 'roa', 'p_e_ratio', 'debt_to_equity', 'piotroski_f_score']
    comparison_metrics = [m for m in comparison_metrics if m in all_stocks_features.columns]

    if comparison_metrics:
        # Calculate mean metrics by region
        regional_means = all_stocks_features.groupby('region')[comparison_metrics].mean()

        fig_regional = go.Figure()

        for i, region in enumerate(regional_means.index):
            fig_regional.add_trace(go.Scatterpolar(
                    r=regional_means.loc[region].values,
                    theta=[m.replace('_', ' ').title() for m in comparison_metrics],
                    fill='toself',
                    name=region,
                    opacity=0.6,
                    ))

        fig_regional.update_layout(
                polar=dict(
                        radialaxis=dict(visible=True,
                                        range=[regional_means.min().min() * 0.8, regional_means.max().max() * 1.2])
                        ),
                title='<b>Financial Metrics by Region</b><br><sup>Radar Chart Comparison</sup>',
                template=PLOTLY_TEMPLATE,
                height=600,
                font=dict(family='Segoe UI, Roboto, Arial'),
                title_font_size=20,
                showlegend=True,
                )
        fig_regional.write_html(financial_metrics_dir / 'regional_comparison.html')
        print(f'    ✓ Saved: regional_comparison.html')

# 2.7 Phase 9.3 Category-Sector Bubble Chart
print('  → phase93_category_sector_bubble_chart.html')
if 'sector' in all_stocks_features.columns:
    # Calculate average coverage by sector for each Phase 9.3 category
    bubble_data = []
    top_sectors = all_stocks_features['sector'].value_counts().head(8).index.tolist()

    for sector in top_sectors:
        sector_df = all_stocks_features[all_stocks_features['sector'] == sector]
        sector_count = len(sector_df)

        for category, features in PHASE93_FEATURE_CATEGORIES.items():
            available_features = [f for f in features if f in sector_df.columns]
            if available_features:
                coverage = sector_df[available_features].notna().mean().mean() * 100
                feature_count = len(available_features)
                bubble_data.append({
                    'Sector': str(sector)[:20],
                    'Category': category,
                    'Coverage %': round(coverage, 1),
                    'Feature Count': feature_count,
                    'Stock Count': sector_count,
                    })

    bubble_df = pd.DataFrame(bubble_data)

    if not bubble_df.empty:
        fig_bubble = px.scatter(
                bubble_df,
                x='Category',
                y='Sector',
                size='Coverage %',
                color='Coverage %',
                color_continuous_scale='RdYlGn',
                hover_data=['Feature Count', 'Stock Count'],
                title='<b>Phase 9.3 Feature Coverage by Sector</b><br><sup>Bubble Size = Coverage Percentage</sup>',
                template=PLOTLY_TEMPLATE,
                )
        fig_bubble.update_layout(
                height=700,
                font=dict(family='Segoe UI, Roboto, Arial'),
                title_font_size=20,
                xaxis_title='Phase 9.3 Category',
                yaxis_title='Sector',
                xaxis_tickangle=45,
                )
        fig_bubble.write_html(financial_metrics_dir / 'phase93_category_sector_bubble_chart.html')
        print(f'    ✓ Saved: phase93_category_sector_bubble_chart.html')

print(f'\n✓ Interactive HTML Visualizations Complete: 7 files generated')
print(f'\n📁 All outputs saved to: {financial_metrics_dir}')

# ============================================================================
# 3. Display Summary Statistics
# ============================================================================
print('\n' + '=' * 80)
print('📊 FINANCIAL METRICS SUMMARY')
print('=' * 80)

# Hypothesis test summary
if hypothesis_results and 'summary' in hypothesis_results:
    print(f"\n🔬 Hypothesis Testing Results:")
    print(f"   Tests performed: {hypothesis_results.get('n_tests', len(test_metrics))}")
    print(f"   Significance level: α = 0.05")
    if 'significant_differences' in hypothesis_results:
        sig_count = sum(1 for v in hypothesis_results['significant_differences'].values() if v)
        print(f"   Significant differences found: {sig_count}/{len(test_metrics)} metrics")

# Quality alerts summary
if quality_alerts:
    print(f"\n⚠️ Data Quality Alerts:")
    if 'outlier_counts' in quality_alerts:
        total_outliers = sum(quality_alerts['outlier_counts'].values())
        print(f"   Total outliers detected: {total_outliers:,}")
    if 'missing_critical' in quality_alerts:
        print(f"   Missing critical values: {len(quality_alerts.get('missing_critical', []))} columns")

# Correlation summary
print(f"\n🔗 Correlation Analysis:")
print(f"   Features analyzed: {len(top_50_features)}")
top_correlations = find_top_correlations(corr_matrix, n_top=5, threshold=0.7)
if not top_correlations.empty:
    print(f"   High correlations (>0.7): {len(top_correlations)} pairs")
    print(f"   Top correlation: {top_correlations.iloc[0]['correlation']:.3f}" if len(top_correlations) > 0 else "")

print('\n✓ Financial Metrics Analytics complete')


FINANCIAL METRICS ANALYTICS

📊 Generating JSON Reports...
  → eda_summary.json
    ✓ Saved: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform\outputs\eda\financial_metrics\eda_summary.json
  → data_quality_alerts.json
    ✓ Saved: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform\outputs\eda\financial_metrics\data_quality_alerts.json
  → metrics_dashboard.json
    ✓ Saved: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform\outputs\eda\financial_metrics\metrics_dashboard.json
  → hypothesis_tests.json
    ✓ Saved: C:\Users\markm\PycharmProjects\Finance_ML_Analytics_Platform\outputs\eda\financial_metrics\hypothesis_tests.json

✓ JSON Reports Complete: 4 files generated

📈 Generating Interactive HTML Visualizations...
  → correlation_heatmap.html
    ✓ Saved: correlation_heatmap.html
  → distributions.html
    ✓ Saved: distributions.html
  → valuation_3d.html
    ✓ Saved: valuation_3d.html
  → region_sector_heatmap.html
    ✓ Saved: region_sector_heatmap.ht

AttributeError: 'list' object has no attribute 'empty'

## Cell 11: Summary & Next Steps

Pipeline execution summary and recommendations for next analysis phases.


In [ ]:
# ============================================================================
# Cell 11: Summary & Next Steps
# ============================================================================

print('=' * 80)
print('ETL DATA EXPLORER - EXECUTION SUMMARY')
print('=' * 80)

# Summary statistics (using engineered features DataFrame)
print(f'\n📊 Dataset Summary:')
print(f'  Total stocks processed:     {len(all_stocks_features):,}')
print(f'  Original columns (ETL):     {len(all_stocks_preprocessed.columns)}')
print(f'  Engineered columns:         {len(all_stocks_features.columns)}')
print(f'  New features added:         {len(all_stocks_features.columns) - len(all_stocks_preprocessed.columns)}')
print(f'  Data source:                {etl_metrics.source_type.upper()}')
print(f'  ETL duration:               {etl_metrics.total_time_sec:.2f}s')
print(f'  Feature engineering time:   {feature_eng_time:.2f}s')

print(f'\n✓ ETL Pipeline Metrics:')
print(f'  Extract time:               {etl_metrics.extract_time_sec:.2f}s')
print(f'  Transform time:             {etl_metrics.transform_time_sec:.2f}s')
print(f'  Load time:                  {etl_metrics.load_time_sec:.2f}s')
print(f'  Quality score:              {etl_metrics.quality_score:.3f}')
print(f'  Validation score:           {etl_metrics.validation_score:.3f}')

print(f'\n✓ Imputation Results:')
print(f'  Strategy:                   {etl_metrics.imputation_strategy}')
print(f'  Missing values (before):    {etl_metrics.missing_values_before_imputation:,}')
print(f'  Missing values (after):     {etl_metrics.missing_values_after_imputation:,}')
imputation_complete = '✓ Complete' if etl_metrics.imputation_completeness else '✗ Incomplete'
print(f'  Completeness:               {imputation_complete}')

print(f'\n✓ Phase 9.3 Feature Engineering Results:')
print(f'  Features present:           {total_phase93_features}/{total_expected}')
print(f'  Coverage:                   {total_phase93_features / total_expected * 100:.1f}%')
print(f'  Categories with features:   {categories_with_features}/{len(PHASE93_FEATURE_CATEGORIES)}')

if 'region' in all_stocks_features.columns:
    print(f'\n📍 Geographic Distribution:')
    for region, count in all_stocks_features['region'].value_counts().head(5).items():
        pct = count / len(all_stocks_features) * 100
        print(f'  {region:20s}: {count:5,} ({pct:5.1f}%)')

if 'sector' in all_stocks_features.columns:
    print(f'\n🏢 Sector Distribution (Top 5):')
    for sector, count in all_stocks_features['sector'].value_counts().head(5).items():
        pct = count / len(all_stocks_features) * 100
        print(f'  {str(sector)[:30]:30s}: {count:5,} ({pct:5.1f}%)')

print(f'\n🎯 Recommended Next Steps:')
print(f'  ✓ Phase 9.3: Feature Engineering - COMPLETE')
print(f'     - {total_phase93_features} Phase 9.3 features engineered')
print(f'     - Momentum, valuation, profitability, quality features integrated')
print(f'  1. Phase 9.4: Classification Modeling')
print(f'     - Event classification for price movements')
print(f'     - Multi-class prediction for market events')
print(f'  2. Phase 9.5: Regression Workflow')
print(f'     - Sector-optimized price target prediction')
print(f'     - Quantile regression for uncertainty bounds')
print(f'  3. Phase 9.6: Model Evaluation & Analytics')
print(f'     - Feature importance analysis')
print(f'     - Model performance by sector/region')
print(f'  4. Phase 9.7: Portfolio Optimization')
print(f'     - Mispricing score calculation')
print(f'     - Stock selection and ranking')
print(f'     - Portfolio construction and backtesting')

print(f'\n' + '=' * 80)
print(f'✓ ETL Data Explorer execution complete!')
print(f'  Timestamp: {pd.Timestamp.now()}')
print(f'  Output: all_stocks_features DataFrame ready for modeling')
print('=' * 80)
